### Installation

In [ ]:
!pip install pandas langchain neo4j

### Importation

In [ ]:
import pandas as pd
from  os import path

In [ ]:
category_df = pd.read_csv('./data/categories.csv')
product_df = pd.read_csv('./data/products.csv')
supplier_df = pd.read_csv('./data/suppliers.csv')

In [ ]:
product_df.head()

### Combine Dataframe

In [ ]:
product_category_df = pd.merge(product_df, category_df, on='categoryID')
product_category_df.head(1)

In [ ]:
product_category_df.columns

#### verifions la somme de toute les valeurs null

In [ ]:
product_category_df.isna().sum()

In [ ]:
supplier_df.head(1)

In [ ]:
product_category_supplier_df = pd.merge(
    product_category_df, 
    supplier_df, 
    on='supplierID',
    how='left'
)
product_category_supplier_df.head(1)

In [ ]:
product_category_supplier_df.shape

In [ ]:
list(product_category_supplier_df.columns)

In [ ]:
product_category_supplier_df.isna().sum()

#### Delete Null value

In [ ]:
product_category_supplier_df["region"] = product_category_supplier_df["region"].replace({pd.NA: "Unknown"})
product_category_supplier_df["fax"] = product_category_supplier_df["fax"].replace({pd.NA: "Unknown"})
product_category_supplier_df["homePage"] = product_category_supplier_df["homePage"].replace({pd.NA: "Unknown"})
product_category_supplier_df.isna().sum()

In [ ]:
product_category_supplier_df.shape

#### Load envs

In [ ]:
from dotenv import dotenv_values, load_dotenv
import os

In [ ]:
%load_ext dotenv
%dotenv

In [ ]:
NEO4J_URI = os.getenv("NEO4J_URI")
NEO4J_USER = os.getenv("NEO4J_USERNAME")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")

In [ ]:
NEO4J_USER, NEO4J_PASSWORD, NEO4J_URI

#### Connect to Neo4j

In [ ]:
import neo4j

In [ ]:
print(neo4j.__version__)

In [ ]:
from neo4j import GraphDatabase

In [ ]:
AUTH = (NEO4J_USER, NEO4J_PASSWORD)

#### Creation du driver

In [ ]:
with GraphDatabase.driver(NEO4J_URI, auth=AUTH) as driver:
    with driver.session() as session:
        result = session.run("RETURN 1 AS number")
        for record in result:
            print(record["number"])

In [ ]:
with GraphDatabase.driver(NEO4J_URI, auth=AUTH) as driver:
    driver.verify_connectivity

In [ ]:
def insert_data(tx, row):

    tx.run("""
    
    MERGE (product:Product {
        productID: $productID
    })
    
    SET
        product.productName = $productName,
        product.supplierID = $supplierID,
        product.categoryID = $categoryID,
        product.quantityPerUnit = $quantityPerUnit,
        product.unitPrice = $unitPrice,
        product.unitsInStock = $unitsInStock,
        product.unitsOnOrder = $unitsOnOrder,
        product.reorderLevel = $reorderLevel,
        product.discontinued = $discontinued

    MERGE (category:Category {
        categoryID: $categoryID
    })

    SET
        category.categoryName = $categoryName,
        category.description = $description

    MERGE (supplier:Supplier {
        supplierID: $supplierID
    })

    SET
        supplier.companyName = $companyName,
        supplier.contactName = $contactName,
        supplier.contactTitle = $contactTitle,
        supplier.address = $address,
        supplier.city = $city,
        supplier.region = $region,
        supplier.postalCode = $postalCode,
        supplier.country = $country,
        supplier.phone = $phone,
        supplier.fax = $fax,
        supplier.homePage = $homePage

    MERGE (product)-[:PART_OF]->(category)
    MERGE (product)-[:SUPPLIED_BY]->(supplier)

    """, **row)

In [ ]:
with driver.session() as session:
    for _, row in product_category_supplier_df.iterrows():
        session.write_transaction(
            insert_data,
            row.to_dict()
        )